In [1]:
import numpy as np
import pandas as pd

dataset = pd.read_csv('train.csv')
dataset.head()

cc  =['blue','dual_sim','four_g','three_g','touch_screen','wifi']

dataset = pd.get_dummies(dataset, columns=cc, dtype=int)


X = dataset.drop('price_range',axis=1).values
y = dataset['price_range'].values

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [2]:
import torch

X_train_tensor = torch.tensor(X_train,dtype=torch.float32)
X_test_tensor = torch.tensor(X_test,dtype=torch.float32)


y_train_tensor = torch.tensor(y_train,dtype=torch.long)
y_test_tensor = torch.tensor(y_test,dtype=torch.long)

from torch.utils.data import Dataset,DataLoader

class MobileDataset(Dataset):
    def __init__(self,X,y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, index):
        return self.X[index],self.y[index]

train_dataset = MobileDataset(X_train_tensor,y_train_tensor)
test_dataset = MobileDataset(X_test_tensor,y_test_tensor)

train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False)

In [3]:
from torch import nn

class MobileClassifier(nn.Module):
    def __init__(self,dim):
        super(MobileClassifier,self).__init__()
        self.layer_1 = nn.Linear(dim,32)
        self.dropout = nn.Dropout(0.2)
        self.layer_2 = nn.Linear(32,16)
        self.layer_3 = nn.Linear(16,8)
        self.layer_4 = nn.Linear(8,4)
        self.relu = nn.ReLU()
    def forward(self,x):
        x = self.relu(self.layer_1(x))
        x = self.dropout(x)
        x = self.relu(self.layer_2(x))
        x = self.dropout(x)
        x = self.relu(self.layer_3(x))
        x = self.layer_4(x)
        return x
i_dim = X_train.shape[1]
model = MobileClassifier(i_dim)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)


In [4]:
epochs = 200

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for X_batch,y_batch in train_loader:
        outputs = model(X_batch)
        loss = criterion(outputs,y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {train_loss/len(train_loader):.4f}')

Epoch [10/200], Loss: 0.3843
Epoch [20/200], Loss: 0.2397
Epoch [30/200], Loss: 0.1783
Epoch [40/200], Loss: 0.1397
Epoch [50/200], Loss: 0.1340
Epoch [60/200], Loss: 0.1047
Epoch [70/200], Loss: 0.0960
Epoch [80/200], Loss: 0.0864
Epoch [90/200], Loss: 0.0961
Epoch [100/200], Loss: 0.0911
Epoch [110/200], Loss: 0.0629
Epoch [120/200], Loss: 0.0830
Epoch [130/200], Loss: 0.0652
Epoch [140/200], Loss: 0.0647
Epoch [150/200], Loss: 0.0608
Epoch [160/200], Loss: 0.0716
Epoch [170/200], Loss: 0.0489
Epoch [180/200], Loss: 0.0602
Epoch [190/200], Loss: 0.0620
Epoch [200/200], Loss: 0.0475


In [5]:
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    test_loss = criterion(test_outputs,y_test_tensor)
    print(f'Test Cross entropy loss: {test_loss.item():.4f}')

Test Cross entropy loss: 0.3090


In [7]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. تفعيل وضع الاختبار لإيقاف الـ Dropout
model.eval()

with torch.no_grad(): # إيقاف حساب التدرجات لتوفير الذاكرة
    # 2. الحصول على التوقعات الخام (Logits)
    # الموديل هون رح يطلع 4 أرقام (مخرجات) لكل موبايل
    test_outputs = model(X_test_tensor)
    
    # 3. السحر الجديد: اختيار الفئة ذات الاحتمالية الأعلى باستخدام argmax
    # dim=1 معناها "دور على أعلى رقم في كل صف (كل موبايل)"
    predictions = torch.argmax(test_outputs, dim=1)
    
    # 4. تحويل التنسورات لـ Numpy عشان نقدر نستخدم مكتبة sklearn للتقييم
    y_test_np = y_test_tensor.numpy()
    predictions_np = predictions.numpy()
    
    # 5. حساب الدقة والمقاييس الأخرى
    accuracy = accuracy_score(y_test_np, predictions_np)
    
    print("=== نتائج تقييم نموذج أسعار الموبايلات ===")
    print(f"دقة النموذج (Accuracy): {accuracy * 100:.2f}%\n")
    
    print("--- تفاصيل التصنيف (Classification Report) ---")
    print(classification_report(y_test_np, predictions_np))
    
    print("--- مصفوفة الارتباك (Confusion Matrix) ---")
    print(confusion_matrix(y_test_np, predictions_np))

=== نتائج تقييم نموذج أسعار الموبايلات ===
دقة النموذج (Accuracy): 93.50%

--- تفاصيل التصنيف (Classification Report) ---
              precision    recall  f1-score   support

           0       0.99      0.88      0.93       105
           1       0.87      0.96      0.91        91
           2       0.93      0.95      0.94        92
           3       0.96      0.96      0.96       112

    accuracy                           0.94       400
   macro avg       0.94      0.94      0.93       400
weighted avg       0.94      0.94      0.94       400

--- مصفوفة الارتباك (Confusion Matrix) ---
[[ 92  13   0   0]
 [  1  87   3   0]
 [  0   0  87   5]
 [  0   0   4 108]]
